In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import cross_val_score
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import SimpleImputer
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import patsy
import geopandas as gpd

In [ ]:
base = "../outputs/tables/"
y = "y_pv"  # change this to "y_pv", "y_storage", "y_chargers", or "y_wind_mw"

def prep_outcomes_per_capita(df, pop_col="total_population", min_pop=1000):
    """Create per-capita + log1p outcomes; filter tiny-pop ZIPs."""
    df = df.copy()
    df[pop_col] = pd.to_numeric(df[pop_col], errors="coerce")
    df = df[df[pop_col].notna() & (df[pop_col] >= min_pop)].copy()

    pop = df[pop_col]
    df["level2_chargers_per_1k"] = df["level2_chargers"] * 1000 / pop
    df["y_level2_chargers"] = np.log1p(df["level2_chargers_per_1k"])

    df["level1_chargers_per_1k"] = df["level1_chargers"] * 1000 / pop
    df["y_level1_chargers"] = np.log1p(df["level1_chargers_per_1k"])

    df["dc_fast_chargers_per_1k"] = df["dc_fast_chargers"] * 1000 / pop
    df["y_dc_fast_chargers"] = np.log1p(df["dc_fast_chargers_per_1k"])

    df["chargers_per_1k"] = df["total_chargers"] * 1000 / pop
    df["y_chargers"] = np.log1p(df["chargers_per_1k"])

    df["pv_kw_per_1k"] = df["PV_system_size_DC"] * 1000 / pop
    df["y_pv"] = np.log1p(df["pv_kw_per_1k"])

    df["storage_mw_per_100k"] = df["storage_capacity_mw"] * 100000 / pop
    df["y_storage"] = np.log1p(df["storage_mw_per_100k"])

    df["wind_mw_per_100k"] = df["wind_capacity_mw"] * 100000 / pop
    df["y_wind_mw"] = np.log1p(df["wind_mw_per_100k"])

    df["log_median_household_income"] = np.log(df["median_household_income"].where(df["median_household_income"] > 0))
    df["log_median_housing_value"] = np.log(df["median_housing_value"].where(df["median_housing_value"] > 0))
    df["combined_nonwhite_share"] = df[["pct_black", "pct_hispanic", "pct_asian"]].sum(axis=1, min_count=1)

    return df

def center_cols(df, cols):
    """Mean-center columns for interaction models."""
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c + "_c"] = df[c] - df[c].mean()
    return df

def run_ols(formula, df, cluster_col=None):
    """Run OLS with HC1 SEs by default; optional clustered SEs."""
    m = smf.ols(formula=formula, data=df)

    if cluster_col is None:
        return m.fit(cov_type="HC1")
    res = m.fit()
    used_idx = res.model.data.row_labels
    groups = df.loc[used_idx, cluster_col].copy()
    groups = groups.fillna("MISSING").astype(str)
    return m.fit(cov_type="cluster", cov_kwds={"groups": groups})

def quick_print(res, title=""):
    """Print and export coefficient table."""
    if title:
        print("
" + "="*80)
        print(title)
        print("="*80)
    print(res.summary().tables[0])
    print(res.summary().tables[1])
    res.summary2().tables[1].to_csv(base + title + ".csv")

def vif_from_formula(formula, df, exclude_prefixes=("C(",), drop_intercept=True):
    y, X = patsy.dmatrices(formula, df, return_type="dataframe")

    if drop_intercept and "Intercept" in X.columns:
        X = X.drop(columns=["Intercept"])

    if exclude_prefixes:
        keep = []
        for col in X.columns:
            if not any(col.startswith(pref) for pref in exclude_prefixes):
                keep.append(col)
        X = X[keep]

    X = X.replace([np.inf, -np.inf], np.nan).dropna()

    vif = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    }).sort_values("VIF", ascending=False)

    return vif



In [ ]:
#### THIS IS JUST FOR CALCULATING DIFFERENT FEATURES
#### ONLY MODIFY TO ADD FEATURES
df = pd.read_csv("../data/processed/combined_der_dataset_w_controls_predictors.csv")
df.drop(columns=['Unnamed: 0'], inplace=True)
df.rename(columns={"ghi_mean_kwh_m2_day_2024":"ghi_mean_kwh_m2_day_2023",
"wind_ws10m_mean_2024": "wind_ws10m_mean_2023",
    "wind_ws50m_mean_2024": "wind_ws50m_mean_2023"}, inplace=True)
# numeric coercion for key vars (safe)
for c in [
    "median_household_income","poverty_rate",
    "pct_black","pct_hispanic","pct_asian",
    "cdd65_2023","hdd65_2023","t2m_mean_c_2023",
    "ghi_mean_kwh_m2_day_2023","total_population",
    "lat","lon"
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# fill utility_type if you plan to use it
if "utility_type" in df.columns:
    df["utility_type"] = df["utility_type"].fillna("POU")

print(df.shape, df.columns[:10])
df["log_median_household_income"] = np.log1p(df["median_household_income"])
df["log_median_housing_value"] = np.log1p(df["median_housing_value"])
df = prep_outcomes_per_capita(df, min_pop=1000)

# keep only rows with core predictors present
core_needed = ["median_household_income", "pct_black", "pct_hispanic", "pct_asian"]
df = df.dropna(subset=[c for c in core_needed if c in df.columns]).copy()

df["stand_cdd65_2023"] = (df["cdd65_2023"] - np.mean(df["cdd65_2023"]))/np.std(df["cdd65_2023"])
df["stand_hdd65_2023"] = (df["hdd65_2023"] - np.mean(df["hdd65_2023"]))/np.std(df["hdd65_2023"])
# make a separate standardized copy for coefficient-comparison plots
### COMMENT AFTER HERE
# df = df.copy()

# cols_to_standardize = [
#     "log_median_household_income",
#     "pct_black",
#     "pct_hispanic",
#     "pct_asian",
#     "poverty_rate",
#     "wind_ws50m_mean_2023"
# ]

# # only include columns that actually exist
# cols_to_standardize = [c for c in cols_to_standardize if c in df.columns]

# scaler = StandardScaler()
# df[cols_to_standardize] = scaler.fit_transform(df[cols_to_standardize])

# # if you want CDD/HDD standardized through sklearn too, use the raw versions instead of the manual stand_ vars:
# climate_cols_to_standardize = [c for c in ["cdd65_2023", "hdd65_2023"] if c in df.columns]
# if climate_cols_to_standardize:
#     df[climate_cols_to_standardize] = StandardScaler().fit_transform(df[climate_cols_to_standardize])

# outcomes available
outcomes = [c for c in ["y_chargers", "y_pv", "y_storage", "y_wind_mw", "y_turbines"] if c in df.columns]
outcomes, df.shape
print(df.columns)

In [ ]:
corr = df[['PV_system_size_DC', 'total_chargers',
       'level1_chargers', 'level2_chargers', 'dc_fast_chargers', 'zev_count',
       'plant_capacity_mw', 'storage_capacity_mw', 'wind_capacity_mw',
       'wind_turbine_count', 'median_household_income', 'poverty_rate',
       'pct_bachelors_plus', 'pct_black', 'pct_hispanic', 'pct_asian',
       'median_housing_value', 'total_population', 'ghi_mean_kwh_m2_day_2023',
       't2m_mean_c_2023', 't2m_max_mean_c_2023', 't2m_min_mean_c_2023',
       't2m_summer_mean_c_2023', 'cdd65_2023', 'hdd65_2023',
       'wind_ws10m_mean_2023', 'wind_ws50m_mean_2023', 'lat', 'lon',
       'overlap_area_m2', 'overlap_share_of_zip',
       'kwh_annual_total', 'log_kwh',
       'area_km2', 'pop_density_km2', 'log_pop_density',
       'log_median_household_income', 'log_median_housing_value',
       'chargers_per_1k', 'y_chargers', 'pv_kw_per_1k', 'y_pv',
       'storage_mw_per_100k', 'y_storage', 'wind_mw_per_100k', 'y_wind_mw',
       'y_turbines', 'any_turbines']].corr()

sns.heatmap(corr)

In [ ]:
#### This is where the variables are defined to be used in the models
income = "log_median_household_income"
race = ["pct_black", "pct_hispanic", "pct_asian"]
race_summary = "combined_nonwhite_share"

controls_common = ["poverty_rate"]

controls_3A = ["cdd65_2023", "hdd65_2023"]
controls_3B = ["t2m_mean_c_2023"]
controls_3C = ["ghi_mean_kwh_m2_day_2023"]
controls_3D = ["wind_ws50m_mean_2023"]
controls_cur = None
if y == "y_pv":
    controls_cur = controls_3C
elif y == "y_storage":
    controls_cur = controls_3A
elif y == "y_chargers":
    controls_cur = controls_3A
elif y == "y_wind_mw":
    controls_cur = controls_3D
print(controls_cur)

ses_bach = ["pct_bachelors_plus"]
ses_house = ["log_median_housing_value"]

utility_fe = "C(utility)"
cluster_utility = "utility"

df["county_geoid"] = df["county_geoid"].astype(str)
county_fe = "C(county_geoid)"

latlon = ["lat","lon"]
lat = ["lat"]
lon = ["lon"]

demand_proxy = "log_kwh"
controls_common, controls_3A, controls_3B, utility_fe

term_labels = {
    "cdd65_2023": "Cooling degree days",
    "hdd65_2023": "Heating degree days",
    "ghi_mean_kwh_m2_day_2023": "Solar irradiance (GHI)",
    "wind_ws10m_mean_2023": "Wind speed (10m)",
    "wind_ws50m_mean_2023": "Wind speed (50m)",
    "poverty_rate": "Poverty rate",
    "pct_bachelors_plus": "% Bachelor's+",
    "pct_black": "% Black",
    "pct_hispanic": "% Hispanic",
    "pct_asian": "% Asian",
    "combined_nonwhite_share": "Combined non-white share",
    "log_median_household_income": "Log median household income",
    "log_median_housing_value": "Log median housing value",
    "log_pop_density": "Log population density",
    "log_kwh": "Log annual electricity demand",
    "y_pv": "Solar PV adoption",
    "y_chargers": "EV charger availability",
    "y_storage": "Storage deployment",
    "y_wind_mw": "Wind capacity",
}



In [ ]:
def build_formula(y, climate_controls, extra_terms=None, fe_terms=None,
                  keepincome=True, demand_proxy=None, controls_common = ["poverty_rate"]):
    rhs = []
    if keepincome:
        rhs.append(income)
    rhs += race
    rhs += list(controls_common)
    if demand_proxy is not None:
        rhs.append(demand_proxy)
    rhs += list(climate_controls)
    if extra_terms:
        rhs += list(extra_terms)
    if fe_terms:
        rhs += list(fe_terms)
    seen = set()
    rhs = [x for x in rhs if not (x in seen or seen.add(x))]

    return f"{y} ~ " + " + ".join(rhs)

Model 1 (Baseline, degree-days climate)

In [ ]:
f1 = build_formula(
    y,
    climate_controls=controls_cur,
    controls_common=["poverty_rate"],
    fe_terms=[])
res1 = run_ols(f1, df)
title = f"{y} | Model 1 baseline (climate controls)"
quick_print(res1, title)
print(vif_from_formula(f1, df).head(30))

Model 2 (SES robustness: add ONE proxy)

In [ ]:
# bachelors proxy
f2a = build_formula(y,
                    climate_controls=controls_cur,
                    extra_terms=ses_bach,
                    keepincome = True,
                    controls_common=["poverty_rate"])
res2a = run_ols(f2a, df)
quick_print(res2a, f"{y} | Model 2 (add bachelors)")
print(vif_from_formula(f2a, df).head(30))

# housing proxy
f2b = build_formula(y,
                    climate_controls=controls_cur,
                    extra_terms=ses_house,
                    keepincome = True,
                    controls_common=["poverty_rate"])
res2b = run_ols(f2b, df)
quick_print(res2b, f"{y} | Model 2 (add housing value)")
print(vif_from_formula(f2b, df).head(30))


Model 3 climate robustness (3B temp-only; 3C PV-only)

In [ ]:
# 3A CDD and HDD only
f3a = build_formula(y, climate_controls=controls_3A)
res3a = run_ols(f3a, df)
quick_print(res3a, f"{y} | Model 3A (HDD + CDD)")
print(vif_from_formula(f3a, df).head(30))

# 3B temp only
f3b = build_formula(y, climate_controls=controls_3B)
res3b = run_ols(f3b, df)
quick_print(res3b, f"{y} | Model 3B (temp only)")
print(vif_from_formula(f3b, df).head(30))

# 3C temp only
f3c = build_formula(y, climate_controls=controls_3C)
res3c = run_ols(f3c, df)
quick_print(res3c, f"{y} | Model 3C (GHI only)")
print(vif_from_formula(f3c, df).head(30))

Model 4 interactions (centered)

In [ ]:
df_int = center_cols(df, [income] + race)

f4 = (
    f"{y} ~ log_median_household_income_c + pct_black_c + pct_hispanic_c + pct_asian_c"
    + (" + " + " + ".join(["poverty_rate"] + controls_cur) if (controls_common or controls_cur) else "")
    + " + log_median_household_income_c:pct_black_c"
    + " + log_median_household_income_c:pct_hispanic_c"
    + " + log_median_household_income_c:pct_asian_c"
)
# can change 3A to 3D
res4 = run_ols(f4, df_int)
quick_print(res4, f"{y} | Model 4 interactions (centered)")
print(vif_from_formula(f4, df_int).head(30))

f4R = (
    f"{y} ~ log_median_household_income_c + pct_black_c + pct_hispanic_c + pct_asian_c"
    + (" + " + " + ".join(controls_common + controls_cur) if (controls_common or controls_cur) else "")
    + " + log_median_household_income_c:pct_black_c"
    + " + log_median_household_income_c:pct_hispanic_c"
    + " + log_median_household_income_c:pct_asian_c"
)

res4R = run_ols(f4R, df_int)
quick_print(res4R, f"{y} | Model 4R interactions (centered)")
print(vif_from_formula(f4R, df_int).head(30))

Model 5 utility fixed effects (+ optional clustered SEs)

In [ ]:
if utility_fe:
    f5 = build_formula(y, climate_controls=controls_cur, fe_terms=[utility_fe])

    res5 = run_ols(f5, df)
    quick_print(res5, f"{y} | Model 5 utility FE")
    print(vif_from_formula(f5, df_int).head(30))

    res5C = run_ols(f5, df, cluster_col="county_geoid")
    quick_print(res5C, f"{y} | Model 5C clustered SEs by county")

Model 6 geography: county FE OR lat/lon

In [ ]:
# 6A Lat/Lon FE
f6a = build_formula(y, climate_controls=[], extra_terms=latlon)
res6a = run_ols(f6a, df)
quick_print(res6a, f"{y} | Model 6A lat and lon")
print(vif_from_formula(f6a, df).head(30))

# 6B County FE
f6b = build_formula(y, climate_controls=[], extra_terms=[county_fe])
res6b = run_ols(f6b, df)
quick_print(res6b, f"{y} | Model 6B county fe")
print(vif_from_formula(f6b, df).head(30))

res6_fe_cluster = run_ols(f1, df, cluster_col="county_geoid")
quick_print(res6_fe_cluster, f"{y} | No County FE + clustered SEs (county)")

Model 7 infrastructure controls

In [ ]:
# outcome -> columns to EXCLUDE from infrastructure controls
exclude_for_y = {
    "y_chargers": ["total_chargers", "level1_chargers", "level2_chargers", "dc_fast_chargers", "chargers_per_1k"],
    "y_pv": ["PV_system_size_DC", "pv_kw_per_1k", "y_pv"],
    "y_storage": ["storage_capacity_mw", "storage_mw_per_100k"],
    "y_wind_mw": ["wind_capacity_mw", "wind_mw_per_100k"],
}

infra_candidates = ["plant_capacity_mw", "storage_capacity_mw", "wind_capacity_mw", "wind_turbine_count", "PV_system_size_DC"]

infra = [c for c in infra_candidates if c in df.columns]
infra = [c for c in infra if c not in exclude_for_y.get(y, [])]

f7 = build_formula(y, climate_controls=controls_cur, extra_terms=infra)
res7 = run_ols(f7, df)
quick_print(res7, f"{y} | Model 7 (infrastructure controls, outcome-safe)")
print(vif_from_formula(f7, df).head(30))



In [ ]:
# Create per-capita infrastructure controls once
pop = df["total_population"].replace(0, np.nan)

if "plant_capacity_mw" in df.columns:
    df["plant_mw_per_100k"] = df["plant_capacity_mw"] * 100000 / pop
if "storage_capacity_mw" in df.columns:
    df["storage_mw_per_100k"] = df["storage_capacity_mw"] * 100000 / pop
if "wind_capacity_mw" in df.columns:
    df["wind_mw_per_100k_ctrl"] = df["wind_capacity_mw"] * 100000 / pop
if "wind_turbine_count" in df.columns:
    df["turbines_per_100k"] = df["wind_turbine_count"] * 100000 / pop

infra_pc = [c for c in ["plant_mw_per_100k", "storage_mw_per_100k", "wind_mw_per_100k_ctrl"] if c in df.columns]
f7pc = build_formula(y, climate_controls=controls_cur, extra_terms=infra_pc)
res7pc = run_ols(f7pc, df)
quick_print(res7pc, f"{y} | Model 7 (per-capita infrastructure controls)")
print(vif_from_formula(f7pc, df).head(30))

Model 8 demand proxy robustness

In [ ]:
if demand_proxy:
    f8 = build_formula(y, climate_controls=controls_cur, extra_terms=[demand_proxy])
    res8 = run_ols(f8, df)
    quick_print(res8, f"{y} | Model 8 add demand proxy")
    print(vif_from_formula(f8, df).head(30))

In [ ]:
# Model 9: most-controlled storage specification with PV as a predictor
if y == "y_storage":
    pop = df["total_population"].replace(0, np.nan)

    if "plant_capacity_mw" in df.columns:
        df["plant_mw_per_100k"] = df["plant_capacity_mw"] * 100000 / pop
    if "wind_capacity_mw" in df.columns:
        df["wind_mw_per_100k_ctrl"] = df["wind_capacity_mw"] * 100000 / pop

    model9_terms = [
        "pct_bachelors_plus",
        "plant_mw_per_100k",
        "wind_mw_per_100k_ctrl",
        "y_pv",
    ]
    f9 = build_formula(
        y,
        climate_controls=controls_cur,
        extra_terms=model9_terms,
        keepincome=True,
        demand_proxy=demand_proxy,
        controls_common=["poverty_rate"],
    )
    res9 = run_ols(f9, df)
    quick_print(res9, f"{y} | Model 9 + pv control (most controlled)")

    vif9 = vif_from_formula(f9, df)
    print(vif9.head(30))
    vif9.to_csv(base + f"{y} | Model 9 + pv control (most controlled) VIF.csv", index=False)
else:
    print("Skipping Model 9 PV-control robustness because it is only reported for storage.")



In [ ]:
zip_gdf = gpd.read_file("../data/raw/boundaries/tl_2023_us_zcta520/tl_2023_us_zcta520.shp")
county_gdf = gpd.read_file("../data/raw/boundaries/tl_2023_us_county/tl_2023_us_county.shp")
ca_counties = county_gdf[county_gdf["STATEFP"] == "06"].copy()
ca_counties = ca_counties.to_crs(zip_gdf.crs)
ca_outline = ca_counties.dissolve()

ca_counties = county_gdf[county_gdf["STATEFP"] == "06"].copy()
print(zip_gdf.columns)
zip_gdf["zip_code"] = zip_gdf["ZCTA5CE20"].astype(str).str.zfill(5)
# or
# zip_gdf["zip_code"] = zip_gdf["GEOID20"].astype(str).str.zfill(5)
df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
ca_zips = set(df["zip_code"].dropna().unique())
zip_gdf = zip_gdf[zip_gdf["zip_code"].isin(ca_zips)].copy()
def make_paper_spatial_figure(
    df,
    zip_gdf,
    res,
    outcome_col,
    id_col="zip_code",
    observed_title=None,
    residual_title=None,
    residual_col="std_residual",
    observed_cmap="viridis",
    residual_cmap="coolwarm",
    figsize=(16, 7),
    hotspot_threshold=None,
    save_path=None,
    ca_outline_gdf=None
):
    """
    Create a paper-style two-panel spatial figure:
    left = observed outcome by ZIP
    right = model residuals by ZIP

    Parameters
    ----------
    df : pd.DataFrame
        Original modeling dataframe.
    zip_gdf : gpd.GeoDataFrame
        ZIP geometry dataframe.
    res : statsmodels results object
        Fitted regression results object.
    outcome_col : str
        Outcome column in df to map, e.g. 'y_pv' or 'y_storage'.
    id_col : str, default 'zip_code'
        Merge key present in both df and zip_gdf.
    observed_title : str or None
        Title for the observed map.
    residual_title : str or None
        Title for the residual map.
    residual_col : str, default 'std_residual'
        Residual column to plot; one of {'residual', 'std_residual'}.
    observed_cmap : str
        Colormap for observed values.
    residual_cmap : str
        Colormap for residuals.
    figsize : tuple
        Figure size.
    hotspot_threshold : float or None
        If provided, outline ZIPs with abs(residual) >= threshold on the residual panel.
        Best used with standardized residuals.
    save_path : str or None
        If provided, save figure to this path.

    Returns
    -------
    merged : gpd.GeoDataFrame
        GeoDataFrame used for plotting.
    fig, axes
        Matplotlib figure and axes.
    """
    # Copy and standardize merge keys
    df2 = df.copy()
    gdf2 = zip_gdf.copy()

    df2[id_col] = df2[id_col].astype(str).str.zfill(5)
    gdf2[id_col] = gdf2[id_col].astype(str).str.zfill(5)

    # Get rows used in model
    used_idx = res.model.data.row_labels
    diag = df2.loc[used_idx, [id_col]].copy()
    diag["fitted"] = res.fittedvalues
    diag["residual"] = res.resid
    diag["std_residual"] = (res.resid - np.mean(res.resid)) / np.std(res.resid)

    # Keep one observed value per ZIP
    observed = df2[[id_col, outcome_col]].drop_duplicates(subset=[id_col]).copy()

    # Merge onto geometry
    merged = gdf2.merge(observed, on=id_col, how="left")
    merged = merged.merge(diag, on=id_col, how="left")

    # California outer outline only
    ca_outline = merged.dissolve()

    # Default titles
    if observed_title is None:
        observed_title = f"{outcome_col} by ZIP"
    if residual_title is None:
        residual_title = f"{outcome_col} model standardized residuals" if residual_col == "std_residual" else f"{outcome_col} model residuals"

    # Residual color scale centered at zero
    vmax = np.nanmax(np.abs(merged[residual_col]))
    if np.isnan(vmax) or vmax == 0:
        vmax = 1.0

    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # Left: observed outcome
    merged.plot(
        column=outcome_col,
        cmap=observed_cmap,
        linewidth=0.1,
        edgecolor="white",
        legend=True,
        ax=axes[0],
        missing_kwds={"color": "lightgray", "label": "No data"}
    )
    ca_outline.plot(
        ax=axes[0],
        facecolor="none",
        edgecolor="black",
        linewidth=1.5,
        zorder=10
    )
    if ca_outline_gdf is not None:
        ca_outline_gdf.boundary.plot(
            ax=axes[0],
            color="black",
            linewidth=1.2,
            zorder=10
        )
    axes[0].set_title(observed_title)
    axes[0].axis("off")

    # Right: residuals
    merged.plot(
        column=residual_col,
        cmap=residual_cmap,
        linewidth=0.1,
        edgecolor="white",
        legend=True,
        vmin=-vmax,
        vmax=vmax,
        ax=axes[1],
        missing_kwds={"color": "lightgray", "label": "No data"}
    )
    ca_outline.plot(
        ax=axes[1],
        facecolor="none",
        edgecolor="black",
        linewidth=1.5,
        zorder=10
    )

    # Optional hotspot outlines
    if hotspot_threshold is not None:
        hotspots = merged[merged[residual_col].abs() >= hotspot_threshold]
        if len(hotspots) > 0:
            hotspots.boundary.plot(ax=axes[1], linewidth=0.8, color="black")
    if ca_outline_gdf is not None:
        ca_outline_gdf.boundary.plot(
            ax=axes[1],
            color="black",
            linewidth=1.2,
            zorder=10
        )
    axes[1].set_title(residual_title)
    axes[1].axis("off")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
    return merged, fig, axes

In [ ]:
import statsmodels.api as sm

def lowess_with_bootstrap_ci(
    df,
    x_col,
    y_col,
    frac=0.35,
    n_boot=300,
    ci=95,
    grid_size=200,
    seed=42,
    xlabel=None,
    ylabel=None,
    title=None,
    save_path=None,
):
    """Scatterplot with LOWESS curve and bootstrap confidence interval."""
    rng = np.random.default_rng(seed)

    d = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    x = d[x_col].to_numpy()
    y = d[y_col].to_numpy()

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    x_grid = np.linspace(x.min(), x.max(), grid_size)
    fit = sm.nonparametric.lowess(y, x, frac=frac, return_sorted=True)
    y_hat = np.interp(x_grid, fit[:, 0], fit[:, 1])

    boot_preds = np.zeros((n_boot, grid_size))
    n = len(d)

    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        xb = x[idx]
        yb = y[idx]

        ord_b = np.argsort(xb)
        xb = xb[ord_b]
        yb = yb[ord_b]

        fit_b = sm.nonparametric.lowess(yb, xb, frac=frac, return_sorted=True)
        boot_preds[b, :] = np.interp(x_grid, fit_b[:, 0], fit_b[:, 1])

    alpha = (100 - ci) / 2
    lower = np.percentile(boot_preds, alpha, axis=0)
    upper = np.percentile(boot_preds, 100 - alpha, axis=0)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, alpha=0.15, s=12)
    ax.plot(x_grid, y_hat, linewidth=2)
    ax.fill_between(x_grid, lower, upper, alpha=0.2)

    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    ax.set_title(title or f"{y_col} vs {x_col} with LOWESS")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return pd.DataFrame({
        "x_grid": x_grid,
        "lowess": y_hat,
        "ci_low": lower,
        "ci_high": upper,
    })

def summarize_lowess_gradient(lowess_df, predictor):
    endpoint_change = lowess_df["lowess"].iloc[-1] - lowess_df["lowess"].iloc[0]
    span = lowess_df["x_grid"].iloc[-1] - lowess_df["x_grid"].iloc[0]
    avg_slope = endpoint_change / span if span != 0 else np.nan
    return pd.DataFrame({
        "predictor": [predictor],
        "outcome": [y],
        "endpoint_change": [endpoint_change],
        "average_slope": [avg_slope],
    })

metric_frames = []
for x_col, xlabel, slug in [
    ("log_median_household_income", "Log median household income", "income"),
    ("combined_nonwhite_share", "Combined non-white share", "nonwhite"),
    ("pct_bachelors_plus", "% Bachelor's+", "education"),
]:
    lowess_df = lowess_with_bootstrap_ci(
        df=df,
        x_col=x_col,
        y_col=y,
        frac=0.35,
        xlabel=xlabel,
        ylabel=term_labels[y],
        title=f"{term_labels[y]} vs {xlabel}",
        save_path=f"../outputs/figures/generated/{slug}_{y}_lowess.png",
    )
    metric_frames.append(summarize_lowess_gradient(lowess_df, x_col))

pd.concat(metric_frames, ignore_index=True).to_csv(
    f"../outputs/tables/{y} | lowess_gradient_metrics.csv", index=False
)



In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

def compare_linear_vs_nonlinear_models(
    df,
    y_col,
    x_cols,
    n_splits=5,
    random_state=42
):
    """
    Compare cross-validated predictive performance of linear and nonlinear models.
    Drops rows with missing y, keeps X imputation inside each pipeline.
    """
    d = df[x_cols + [y_col]].replace([np.inf, -np.inf], np.nan).copy()

    # crucial fix: drop rows with missing outcome
    d = d[d[y_col].notna()].copy()

    X = d[x_cols]
    y = d[y_col]

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    models = {
    "Linear": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Polynomial deg 2": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("model", LinearRegression())
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            random_state=random_state,
            n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBRegressor(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=random_state,
            n_jobs=-1
        ))
    ])
}

    rows = []
    for name, model in models.items():
        r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
        rmse = -cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error"
        )

        rows.append({
            "model": name,
            "n_obs": len(d),
            "cv_r2_mean": r2.mean(),
            "cv_r2_sd": r2.std(),
            "cv_rmse_mean": rmse.mean(),
            "cv_rmse_sd": rmse.std()
        })

    return pd.DataFrame(rows).sort_values("cv_r2_mean", ascending=False)

predictors = [
    "log_median_household_income",
    "pct_black",
    "pct_hispanic",
    "pct_asian",
    "poverty_rate",
] + controls_cur

compare_linear_vs_nonlinear_models(
    df=df,
    y_col=y,
    x_cols=predictors
)

In [ ]:
def plot_model_comparison(results_df, title=None, save_path=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(results_df["model"], results_df["cv_r2_mean"], yerr=results_df["cv_r2_sd"], capsize=4)
    ax.set_ylabel("Cross-validated $R^2$")
    ax.set_title(title or "Predictive performance comparison")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

res_compare = compare_linear_vs_nonlinear_models(
    df=df,
    y_col=y,
    x_cols=predictors
)

plot_model_comparison(
    res_compare,
    title=f"{term_labels[y]} prediction: linear vs nonlinear models",
    save_path=f"../outputs/figures/{term_labels[y]}_model_comparison.png"
)

In [ ]:
def safe_slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_").lower()

def export_spatial_maps_for_models(
    df,
    zip_gdf,
    outcome_col,
    outcome_title,
    model_results,
    id_col="zip_code",
    output_dir="../outputs/figures/",
    hotspot_threshold=2.0,
    ca_outline_gdf=None,
):
    """
    Generate and save a paper-style spatial figure for each fitted model result.

    Parameters
    ----------
    df : pd.DataFrame
        Original modeling dataframe.
    zip_gdf : gpd.GeoDataFrame
        ZIP geometry dataframe.
    outcome_col : str
        Outcome column in df, e.g. 'y_storage'.
    outcome_title : str
        Human-readable title prefix, e.g. 'Storage adoption by ZIP'.
    model_results : dict[str, statsmodels result]
        Mapping from model label to fitted result object.
    id_col : str
        Merge key.
    output_dir : str
        Folder to save figures into.
    hotspot_threshold : float or None
        Threshold for outlining large residual ZIPs.

    Returns
    -------
    dict
        Mapping from model label to saved path.
    """
    os.makedirs(output_dir, exist_ok=True)
    saved = {}

    for model_name, res in model_results.items():
        fname = f"{outcome_col}_{safe_slug(model_name)}_spatial_figure.png"
        save_path = os.path.join(output_dir, fname)

        gdf_out, fig, axes = make_paper_spatial_figure(
            df=df,
            zip_gdf=zip_gdf,
            res=res,
            outcome_col=outcome_col,
            id_col=id_col,
            observed_title=outcome_title,
            residual_title=f"{outcome_title} — {model_name} standardized residuals",
            hotspot_threshold=hotspot_threshold,
            save_path=save_path,
            ca_outline_gdf=ca_outline_gdf,
        )

        saved[model_name] = save_path
        plt.close(fig)

    return saved

In [ ]:
import os
import re
models = {
    "Model 1 baseline (climate controls)": res1,
    "Model 2a add bachelors": res2a,
    "Model 2b add housing value": res2b,
    "Model 3a HDD + CDD": res3a,
    "Model 3b temp only": res3b,
    "Model 3c GHI only": res3c,
    "Model 4 interactions": res4,
    "Model 4R interactions reduced": res4R,
    "Model 5 utility FE": res5,
    "Model 5C clustered SEs by county": res5C,
    "Model 6a lat lon": res6a,
    "Model 6b county FE": res6b,
    "Model 6 FE clustered": res6_fe_cluster,
    "Model 7 infrastructure controls": res7,
    "Model 7pc per-capita infrastructure controls": res7pc,
    "Model 8 demand proxy": res8,
}
if y == "y_storage" and 'res9' in globals():
    models["Model 9 + pv control (most controlled)"] = res9

saved_maps = export_spatial_maps_for_models(
    df=df,
    zip_gdf=zip_gdf,
    outcome_col=y,
    outcome_title=f"{term_labels[y]} by ZIP",
    model_results=models,
    id_col="zip_code",
    output_dir=f"../outputs/figures/{y[2:]}_maps/",
    hotspot_threshold=2.0,
    ca_outline_gdf=ca_outline,
)
saved_maps

